In [1]:
# HW3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# Read the Excel file and calculate for monthly returns
df = pd.read_excel('HSI.xlsx', index_col=0)
df_interpolated = df.interpolate(method='spline', order=3)
monthly_df = df_interpolated.resample('ME').last()
monthly_returns = monthly_df.pct_change()

# Drop the first row which contains NaN values
monthly_returns = monthly_returns.iloc[1:]

In [2]:
monthly_returns

,10 HK,101 HK,1038 HK,11 HK,12 HK,14 HK,142 HK,16 HK,17 HK,179 HK,...,1299 HK,151 HK,322 HK,1928 HK,135 HK,27 HK,2319 HK,823 HK,1 HK,1113 HK
Date,,,,,,,,,,,,,,,,,,,,,
2001-03-31,-0.195266,0.000000,-0.095745,-0.024259,-0.167727,-0.171329,-0.178786,-0.143678,-0.239577,-0.049808,...,NaN,NaN,0.238532,NaN,0.018519,-0.090909,NaN,NaN,-0.134918,NaN
2001-04-30,0.085089,-0.034441,0.019478,0.014210,-0.087100,-0.112488,-0.025988,-0.030785,0.018404,0.215794,...,NaN,NaN,-0.340788,NaN,0.291527,0.077359,NaN,NaN,0.086120,NaN
2001-05-31,-0.003877,0.011861,-0.026799,-0.057590,0.004368,-0.020635,-0.116926,0.007526,-0.018071,-0.101211,...,NaN,NaN,0.292228,NaN,0.238844,0.039580,NaN,NaN,-0.042689,NaN
2001-06-30,0.027211,0.052941,0.067194,-0.075145,-0.049473,-0.033981,0.018361,-0.034364,-0.049981,-0.210332,...,NaN,NaN,0.008696,NaN,-0.136364,-0.035714,NaN,NaN,0.000000,NaN
2001-07-31,0.019868,-0.039106,-0.070370,0.093750,0.063589,-0.080402,-0.083065,-0.024911,-0.110455,0.065421,...,NaN,NaN,-0.094828,NaN,-0.052632,0.000000,NaN,NaN,-0.091175,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015-11-30,-0.060391,-0.045216,-0.058905,-0.014727,-0.038268,-0.049347,-0.092279,-0.082772,-0.070999,-0.012302,...,0.015351,-0.057276,-0.157658,-0.069272,0.070978,-0.144465,-0.180611,0.015070,-0.045070,-0.072477
2015-12-31,-0.047259,-0.028634,0.057437,0.049822,-0.005236,-0.030534,0.066390,-0.017838,-0.007772,-0.055160,...,0.006479,-0.052545,-0.014260,0.013359,0.016200,0.072368,0.024311,-0.015907,0.028515,-0.002967
2016-01-31,-0.152778,-0.191610,0.013928,-0.128814,-0.115789,-0.055118,0.040856,-0.104701,-0.177546,-0.135593,...,-0.080472,-0.119584,-0.204340,0.009416,-0.160870,-0.014315,-0.150316,-0.044181,-0.078872,-0.173611


In [3]:
# Smart beta schemes objective functions
def mdr(weights, cov):
    portfolio_vol = np.sqrt(np.dot(weights.transpose(), np.dot(cov, weights)))
    weighted_sum_vol = np.sum(weights * np.sqrt(np.diag(cov)))
    return -weighted_sum_vol / portfolio_vol

def gmv(weights, cov):
    return np.dot(weights.transpose(), np.dot(cov, weights))

def msr(weights, returns, cov):
    portfolio_return = np.dot(weights, returns)
    portfolio_vol = np.sqrt(np.dot(weights.transpose(), np.dot(cov, weights)))
    return -portfolio_return / portfolio_vol

In [4]:
# Optimization function
def optimize_portfolio(objective_func, data, max_weight=0.1, *args):
    returns = data.mean()
    cov = data.cov()
    n = cov.shape[0]
    
    # Initial weights
    weights = np.ones(n) / n
    
    # Constraint: sum of weights = 1
    sum_constraint = {'type': 'eq', 'fun': lambda x: np.sum(x) - 1}
    
    # Bounds: between 0 and max_weight for each stock
    bounds = [(0, max_weight) for _ in range(n)]
    
    # Optimize
    if objective_func == msr:
        result = minimize(objective_func, weights, args=(returns, cov), 
                         method='SLSQP', constraints=[sum_constraint], 
                         bounds=bounds, tol=1e-10)
    else:
        result = minimize(objective_func, weights, args=(cov,), 
                         method='SLSQP', constraints=[sum_constraint], 
                         bounds=bounds, tol=1e-10)
    
    return result.x

In [5]:
# 10% upper bound
max_weight = 0.1 

# a. Maximum Diversification Ratio (MDR)
mdr_weights = optimize_portfolio(mdr, monthly_returns, max_weight)

# b. Global Minimum Variance (GMV)
gmv_weights = optimize_portfolio(gmv, monthly_returns, max_weight)

# c. Maximum Sharpe Ratio (MSR)
msr_weights = optimize_portfolio(msr, monthly_returns, max_weight)

# Create a DataFrame to display the weights
weight_df = pd.DataFrame({
    'MDR': mdr_weights,
    'GMV': gmv_weights,
    'MSR': msr_weights
}, index=monthly_returns.columns)

In [12]:
weight_df

,MDR,GMV,MSR
10 HK,0.000000e+00,2.474857e-18,1.790755e-15
101 HK,0.000000e+00,0.000000e+00,9.061556e-16
1038 HK,1.000000e-01,1.000000e-01,1.000000e-01
11 HK,0.000000e+00,6.269833e-02,2.619073e-16
12 HK,0.000000e+00,0.000000e+00,2.423087e-15
...,...,...,...
27 HK,4.694383e-04,0.000000e+00,3.078126e-02
2319 HK,1.410650e-15,0.000000e+00,2.850621e-14
823 HK,6.299818e-02,1.000000e-01,1.000000e-01
1 HK,0.000000e+00,0.000000e+00,1.371985e-15
